# Word/Line-level TrOCR — Kaggle GPU Training (document OCR)

Fine-tunes `microsoft/trocr-base-handwritten` on **synthetic Nepali WORD images** so the model can read joined handwriting **with matras/conjuncts/punctuation** — unlike the single-character CRNN/TrOCR.

**No dataset attachment needed** — the synthetic data is generated inside this notebook.

Settings panel: **Accelerator = GPU T4**, **Internet = ON** (needed to clone the repo, apt-get Devanagari fonts, and download the pretrained TrOCR weights).

Run all cells top to bottom. Trained weights + logs land in `/kaggle/working/artifacts`.

Code: repo branch `ml` (push your latest work there first).

In [ ]:
# 1. Confirm GPU is on
import torch
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4'
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code (branch: ml). transformers/torchvision/Pillow are preinstalled on Kaggle.
import os
if not os.path.isdir('/kaggle/working/Devnagari_Handwriting_Recognition'):
    !git clone -b ml https://github.com/Sanskriti-poudel/Devnagari_Handwriting_Recognition.git /kaggle/working/Devnagari_Handwriting_Recognition
%cd /kaggle/working/Devnagari_Handwriting_Recognition
!git log --oneline -1

In [ ]:
# 3. Install Devanagari fonts (Kaggle is Linux — no Nirmala/Mangal). The generator
#    auto-detects these once installed. Without them it renders empty boxes.
!apt-get -qq update && apt-get -qq install -y fonts-deva fonts-lohit-deva fonts-indic fonts-noto-core > /dev/null 2>&1
import glob
found = [f for f in glob.glob('/usr/share/fonts/**/*.ttf', recursive=True)
         if any(k in os.path.basename(f).lower() for k in ['devanagari','lohit-deva','gargi','samanata','kalimati','nakula','sahadeva'])]
print(f'{len(found)} Devanagari font file(s) found, e.g.:')
for f in found[:8]:
    print('  ', f)
assert found, 'No Devanagari fonts installed — check Internet is ON and apt-get succeeded.'

In [ ]:
# 4. Generate the synthetic training set (mixed real Nepali words + random syllables).
#    ~8k images is a reasonable first pretrain; bump --n for more. Fast (CPU, a few min).
!python data/generate_synth.py --out Datasets/synth --n 8000 --real-ratio 0.5 --seed 42
import os
print('images:', len(os.listdir('Datasets/synth/images')), '| labels.csv exists:', os.path.exists('Datasets/synth/labels.csv'))

In [ ]:
# 4b. SANITY: eyeball a few generated images + labels. The glyphs must render as
#     real Devanagari (not tofu boxes). If you see boxes, the font install failed.
import csv, matplotlib.pyplot as plt
from PIL import Image
rows = list(csv.DictReader(open('Datasets/synth/labels.csv', encoding='utf-8')))[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 5))
for ax, r in zip(axes.ravel(), rows):
    ax.imshow(Image.open(os.path.join('Datasets/synth', r['image_path'])), cmap='gray')
    ax.set_title(r['text']); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 5. Fine-tune the WORD-level TrOCR on the synthetic data.
#    batch_size=4 fits trocr-base on a 16GB T4 (drop to 2 if OOM). Output tee'd to a log.
#    Weights -> models/trocr/checkpoints_words (separate from the single-char checkpoint).
#    Knobs: TROCR_BATCH_SIZE, TROCR_EPOCHS, TROCR_LR, TROCR_NUM_WORKERS.
import os
os.environ['TROCR_BATCH_SIZE'] = '4'
os.environ['TROCR_EPOCHS'] = '6'
os.environ['TROCR_NUM_WORKERS'] = '2'
os.makedirs('/kaggle/working/artifacts', exist_ok=True)
!python -u models/trocr/train_words.py --labels Datasets/synth/labels.csv 2>&1 | tee /kaggle/working/artifacts/train_words.log

In [ ]:
# 5b. SANITY: run the trained model on a few held-out synthetic lines. Predictions
#     should be real multi-glyph Devanagari close to the ground truth (not empty/garbage).
import sys, csv, os
sys.path.insert(0, 'models/trocr'); sys.path.insert(0, '.')
os.environ['TROCR_WORDS_CHECKPOINT'] = 'models/trocr/checkpoints_words'
from predict_words import predict_line
rows = list(csv.DictReader(open('Datasets/synth/labels.csv', encoding='utf-8')))[-6:]
for r in rows:
    out = predict_line(os.path.join('Datasets/synth', r['image_path']))
    print(f"gt={r['text']!r:30} pred={out['text']!r:30} conf={out['confidence']}")

In [ ]:
# 6. Save the trained weights + logs to /kaggle/working/artifacts so they download
#    with the notebook version. Download the artifacts/ folder afterwards.
import shutil, os
os.makedirs('/kaggle/working/artifacts', exist_ok=True)
if os.path.exists('logs/trocr_words_training.csv'):
    shutil.copy('logs/trocr_words_training.csv', '/kaggle/working/artifacts/')
if os.path.isdir('models/trocr/checkpoints_words'):
    shutil.copytree('models/trocr/checkpoints_words',
                    '/kaggle/working/artifacts/checkpoints_words', dirs_exist_ok=True)
    print('saved checkpoint dir')
print(os.listdir('/kaggle/working/artifacts'))